In [ ]:
!pip install kaggle networkx python-louvain

In [ ]:
# Cell 1: Install kagglehub (tool to download public Kaggle datasets)
!pip install -q kagglehub
print("kagglehub installed successfully")


kagglehub installed successfully


In [ ]:
# Cell 2: Download the CAVIAR dataset and see what files it contains
import kagglehub
import os

path = kagglehub.dataset_download("chiragtagadiya/caviar")
print("Downloaded to folder:", path)
print()
print("Files inside:")
for root, dirs, files in os.walk(path):
    for f in sorted(files):
        full = os.path.join(root, f)
        size_kb = os.path.getsize(full) / 1024
        print(f"  {f}  ({size_kb:.1f} KB)")


100%|██████████| 5.24k/5.24k [00:00<00:00, 6.46MB/s]

Extracting files...
Downloaded to folder: /root/.cache/kagglehub/datasets/chiragtagadiya/caviar/versions/1

Files inside:
  phase1.csv  (0.6 KB)
  phase10.csv  (3.9 KB)
  phase11.csv  (3.7 KB)
  phase2.csv  (1.3 KB)
  phase3.csv  (2.4 KB)
  phase4.csv  (2.5 KB)
  phase5.csv  (2.3 KB)
  phase6.csv  (1.7 KB)
  phase7.csv  (2.9 KB)
  phase8.csv  (3.8 KB)
  phase9.csv  (2.6 KB)


In [ ]:
# Cell 3 (fix 2): Force fresh download + locate and inspect phase1.csv
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download("chiragtagadiya/caviar", force_download=True)
print("Data folder:", path)

# Search everywhere inside the folder for phase.csv
found = None
for root, dirs, files in os.walk(path):
    if "phase1.csv" in files:
        found = os.path.join(root, "phase1.csv")
print("Found file at:", found)

df = pd.read_csv(found)

print("Shape (rows x columns):", df.shape)
print()
print("Column names:", list(df.columns))
print()
print("First 5 rows:")
print(df.head())


Using Colab cache for faster access to the 'caviar' dataset.
Data folder: /kaggle/input/caviar
Found file at: /kaggle/input/caviar/CAVIAR/phase1.csv
Shape (rows x columns): (15, 16)

Column names: ['Unnamed: 0', '1', '4', '89', '83', '3', '5', '88', '85', '90', '2', '7', '54', '6', '64', '8']

First 5 rows:
   Unnamed: 0  1  4  89  83  3  5  88  85  90  2  7  54  6  64  8
0           1  0  1   4   0  4  2   2   9   1  2  0   2  0   1  1
1           4  0  0   0   0  0  0   0   0   0  0  0   0  0   0  0
2          89  1  0   0   0  0  0   0   0   0  0  3   0  0   0  0
3          83  1  0   0   0  0  0   0   0   0  0  0   0  5   0  0
4           3  2  0   0   0  0  0   1   0   0  0  0   0  0   0  0


In [ ]:
# Cell 4: Build combined NetworkX graph from all 11 phases
import kagglehub
import pandas as pd
import networkx as nx
import os

path = kagglehub.dataset_download("chiragtagadiya/caviar")

G_combined = nx.Graph()   # one big undirected graph

for i in range(1, 12):
    # find and load this phase's csv
    fname = os.path.join(path, "CAVIAR", f"phase{i}.csv")
    df = pd.read_csv(fname, index_col=0)

    # add every person appearing in this phase
    for node_id in df.index:
        G_combined.add_node(int(node_id), source="caviar")

    # add edges: only where communication > 0
    for r in range(len(df)):
        for c in range(len(df.columns)):
            w = df.iloc[r, c]
            if w > 0:
                a, b = int(df.index[r]), int(df.columns[c])
                if G_combined.has_edge(a, b):
                    # keep the strongest observed value across phases
                    if w > G_combined[a][b]["weight"]:
                        G_combined[a][b]["weight"] = int(w)
                else:
                    G_combined.add_edge(a, b, weight=int(w))

    print(f"Phase {i}: {df.shape[0]} people")

print()
print("=== COMBINED NETWORK (all 11 phases merged) ===")
print("Total unique individuals (nodes):", G_combined.number_of_nodes())
print("Total unique communication links (edges):", G_combined.number_of_edges())


Using Colab cache for faster access to the 'caviar' dataset.
Phase 1: 15 people
Phase 2: 24 people
Phase 3: 33 people
Phase 4: 33 people
Phase 5: 32 people
Phase 6: 27 people
Phase 7: 36 people
Phase 8: 42 people
Phase 9: 34 people
Phase 10: 42 people
Phase 11: 41 people

=== COMBINED NETWORK (all 11 phases merged) ===
Total unique individuals (nodes): 107
Total unique communication links (edges): 214


In [ ]:
# Cell 5: Compute degree and betweenness centrality
import networkx as nx

degree_cent = nx.degree_centrality(G_combined)
between_cent = nx.betweenness_centrality(G_combined, weight="weight")

top_degree = sorted(degree_cent.items(), key=lambda x: -x[1])[:10]
top_between = sorted(between_cent.items(), key=lambda x: -x[1])[:10]

print("=== TOP 10 BY DEGREE CENTRALITY ===")
print("(most direct contacts)")
for rank, (node, score) in enumerate(top_degree, 1):
    print(f"{rank:2}. Person {node:4}  score={score:.3f}")

print()
print("=== TOP 10 BY BETWEENNESS CENTRALITY ===")
print("(most often on paths between others -> potential intermediaries)")
for rank, (node, score) in enumerate(top_between, 1):
    print(f"{rank:2}. Person {node:4}  score={score:.3f}")


=== TOP 10 BY DEGREE CENTRALITY ===
(most direct contacts)
 1. Person    1  score=0.538
 2. Person    3  score=0.283
 3. Person   12  score=0.245
 4. Person   76  score=0.151
 5. Person   87  score=0.151
 6. Person   83  score=0.113
 7. Person   37  score=0.104
 8. Person   41  score=0.094
 9. Person   89  score=0.085
10. Person   85  score=0.085

=== TOP 10 BY BETWEENNESS CENTRALITY ===
(most often on paths between others -> potential intermediaries)
 1. Person    1  score=0.499
 2. Person    3  score=0.273
 3. Person   12  score=0.267
 4. Person    7  score=0.178
 5. Person   87  score=0.116
 6. Person   89  score=0.105
 7. Person   76  score=0.093
 8. Person   14  score=0.086
 9. Person   24  score=0.079
10. Person   32  score=0.076


In [ ]:
# Cell 6: Detect communities in the CAVIAR network
import networkx as nx
from networkx.algorithms.community import louvain_communities

communities = louvain_communities(G_combined, weight="weight", seed=42)
communities = sorted(communities, key=len, reverse=True)

print(f"Number of communities detected: {len(communities)}")
print()
for i, comm in enumerate(communities, 1):
    members = sorted(comm)
    print(f"Community {i} ({len(members)} people): {members}")


Number of communities detected: 7

Community 1 (52 people): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 15, 19, 20, 22, 28, 31, 32, 34, 35, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 62, 64, 67, 68, 69, 70, 74, 77, 83, 84, 85, 86, 88, 89, 90, 97, 98, 99, 100, 106, 108, 109]
Community 2 (20 people): [13, 21, 27, 37, 38, 39, 40, 41, 42, 43, 44, 45, 81, 87, 91, 92, 93, 94, 95, 105]
Community 3 (16 people): [12, 14, 16, 17, 18, 23, 24, 25, 26, 29, 33, 58, 65, 66, 75, 80]
Community 4 (10 people): [59, 61, 71, 72, 73, 76, 78, 79, 101, 102]
Community 5 (5 people): [30, 36, 46, 82, 96]
Community 6 (2 people): [63, 107]
Community 7 (2 people): [103, 104]


In [ ]:
# Cell 7: Download IL-TUR L-NER task and show its structure
!pip install -q datasets

from datasets import load_dataset

ner = load_dataset("Exploration-Lab/IL-TUR", "ln-ner")

print("Splits available:", list(ner.keys()))
for split in ner:
    print(f"{split}: {len(ner[split])} examples")
    print("Fields/columns:", ner[split].column_names)


README.md:   0%|          | 0.00/37.9k [00:00<?, ?B/s]

DatasetNotFoundError: Dataset 'Exploration-Lab/IL-TUR' is a gated dataset on the Hub. You must be authenticated to access it.

In [ ]:
# Cell 7b: Log in to Hugging Face securely
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
# Cell 7c: Log in to Hugging Face, then load IL-TUR L-NER
from huggingface_hub import notebook_login
from datasets import load_dataset

notebook_login()   # paste your hf_... token in the box that appears

ner = load_dataset("Exploration-Lab/IL-TUR", "ln-ner")

print("Splits available:", list(ner.keys()))
for split in ner:
    print(f"{split}: {len(ner[split])} examples")
    print("Fields/columns:", ner[split].column_names)


ValueError: BuilderConfig 'ln-ner' not found. Available: ['lner', 'rr', 'cjpe', 'bail', 'lsi', 'pcr', 'summ', 'lmt']

In [ ]:
# Cell 7d: Load IL-TUR with the correct task name "lner"
from datasets import load_dataset

ner = load_dataset("Exploration-Lab/IL-TUR", "lner")

print("Splits available:", list(ner.keys()))
for split in ner:
    print(f"{split}: {len(ner[split])} examples")
    print("Fields/columns:", ner[split].column_names)


lner/fold_1-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  584kB            

lner/fold_1-00000-of-00001.parquet: downloading bytes:           |  0.00B            

lner/fold_2-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  589kB            

lner/fold_2-00000-of-00001.parquet: downloading bytes:           |  0.00B            

lner/fold_3-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  612kB            

lner/fold_3-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating fold_1 split:   0%|          | 0/35 [00:00<?, ? examples/s]

Generating fold_2 split:   0%|          | 0/35 [00:00<?, ? examples/s]

Generating fold_3 split:   0%|          | 0/35 [00:00<?, ? examples/s]

Splits available: ['fold_1', 'fold_2', 'fold_3']
fold_1: 35 examples
Fields/columns: ['id', 'text', 'spans']
fold_2: 35 examples
Fields/columns: ['id', 'text', 'spans']
fold_3: 35 examples
Fields/columns: ['id', 'text', 'spans']


In [ ]:
# Cell 8: Inspect one real annotated example + count all entity labels
from collections import Counter

ex = ner["fold_1"][0]

print("Record ID:", ex["id"])
print()
print("=== REAL COURT JUDGMENT TEXT ===")
print(ex["text"][:800])   # first 800 characters
print("...")
print()
print("=== ANNOTATED ENTITIES IN THIS DOCUMENT ===")
for span in ex["spans"]:
    start = span.get("start", "?")
    end = span.get("end", "?")
    label = span.get("label", "?")
    snippet = ex["text"][start:end] if isinstance(start, int) else span
    print(f"  [{label}]  \"{snippet}\"  (chars {start}-{end})")

# count label types across all folds
label_counts = Counter()
for split in ner:
    for e in ner[split]:
        for s in e["spans"]:
            label_counts[s.get("label", "?")] += 1

print()
print("=== ALL ENTITY LABELS ACROSS 105 EXAMPLES ===")
for label, count in label_counts.most_common():
    print(f"  {label}: {count}")


Record ID: 115651329

=== REAL COURT JUDGMENT TEXT ===
REPORTABLE IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 92/2015 JAGE RAM & ORS. ..Appellants Versus STATE OF HARYANA ..Respondent J U D G M E N T R. BANUMATHI, J. This appeal is preferred against the judgment dated 19.8.2011 passed by the High Court of Punjab and Haryana in Criminal Appeal No.181 SB of 2000, whereby the High Court partly allowed the appeal filed by the appellants thereby confirming the conviction of the appellants with certain modifications. 2. Briefly stated, case of the prosecution is that on the fateful day i.e. 18.11.1994, at about 8.00 A.M. in the morning the complainant Jagdish (PW-5) along with his two sons namely Sukhbir and Mange Ram (PW-6) were busy in cutting pullas (reeds) from the dola of their field. At that time, Jage Ram
...

=== ANNOTATED ENTITIES IN THIS DOCUMENT ===
  [1]  "STATE OF HARYANA"  (chars 137-153)
  [10]  "19.8.2011"  (chars 252-261)
  [7]  "High Co

In [ ]:
# Cell 9: Look for label name definitions inside the dataset metadata
print("Features of 'spans' column:")
print(ner["fold_1"].features["spans"])


Features of 'spans' column:
List({'start': Value('int64'), 'end': Value('int64'), 'label': ClassLabel(names=['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO'])})


In [ ]:
# Cell 10: Summarize entities across all 105 records
import pandas as pd

LABEL_NAMES = ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT',
               'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']

# count each label type per record
rows = []
examples = {}   # first seen text snippet per label
for split in ner:
    for rec in ner[split]:
        counts = {name: 0 for name in LABEL_NAMES}
        for s in rec["spans"]:
            name = LABEL_NAMES[s["label"]]
            counts[name] += 1
            if name not in examples:
                examples[name] = rec["text"][s["start"]:s["end"]]
        row = {"record_id": rec["id"], "source_split": split}
        row.update(counts)
        rows.append(row)

df_entities = pd.DataFrame(rows)
print(f"Entity summary table: {df_entities.shape[0]} records x {len(LABEL_NAMES)} entity types")
print(df_entities.head(5))
print()

print("=== ONE REAL EXAMPLE PER ENTITY TYPE ===")
for i, name in enumerate(LABEL_NAMES):
    print(f"  {name:9} e.g. \"{examples.get(name, '(none found)')}\"")


Entity summary table: 105 records x 12 entity types
   record_id source_split  APP  RESP  A.COUNSEL  R.COUNSEL  JUDGE  WIT  AUTH  \
0  115651329       fold_1    9     2          2          4      5   26     0   
1   37849282       fold_1    3     5          0          0      3    0     0   
2     975074       fold_1    1     2          1          0      3    0     4   
3  189525449       fold_1   52     2          1          1      8   10     0   
4     736324       fold_1    1     4          2          1      3    0     9   

   COURT  STAT  PREC  DATE  CASENO  
0      8    26    10    10       2  
1      1     2     0    40      12  
2     45    25    11     5       5  
3      8    15     6     7       2  
4     28     3     4    30      26  

=== ONE REAL EXAMPLE PER ENTITY TYPE ===
  APP       e.g. "Jage Ram"
  RESP      e.g. "STATE OF HARYANA"
  A.COUNSEL e.g. "Vibha Datta Makhija"
  R.COUNSEL e.g. "Ajay Bansal"
  JUDGE     e.g. "V. Gopala Gowda"
  WIT       e.g. "Jagdish"
  AUTH 

In [ ]:
# Cell 11: Export graph to JSON and build the D3.js HTML visualization
import json
import networkx as nx
from networkx.algorithms.community import louvain_communities

# --- rebuild communities deterministically ---
communities = sorted(louvain_communities(G_combined, weight="weight", seed=42),
                     key=len, reverse=True)
node_to_comm = {}
for ci, comm in enumerate(communities):
    for n in comm:
        node_to_comm[n] = ci

degree_cent = nx.degree_centrality(G_combined)
between_cent = nx.betweenness_centrality(G_combined, weight="weight")

# --- export to JSON ---
graph_data = {
    "nodes": [
        {"id": int(n),
         "degree": round(degree_cent[n], 3),
         "betweenness": round(between_cent[n], 4),
         "community": node_to_comm[n]}
        for n in G_combined.nodes()
    ],
    "links": [
        {"source": int(u), "target": int(v), "weight": d["weight"]}
        for u, v, d in G_combined.edges(data=True)
    ]
}
with open("caviar_graph.json", "w") as f:
    json.dump(graph_data, f)
print(f"Exported {len(graph_data['nodes'])} nodes, {len(graph_data['links'])} links")

# --- build HTML with embedded data ---
html = """<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>CAVIAR Network Explorer</title>
<script src="https://d3js.org/d3.v7.min.js"></script>
<style>
  body { margin:0; font-family:sans-serif; background:#111; color:#eee; }
  #panel { position:fixed; right:0; top:0; width:300px; height:100%;
           background:#1b1b1b; border-left:2px solid #444; padding:15px;
           box-sizing:border-box; overflow-y:auto; }
  .node { stroke:#000; stroke-width:1px; cursor:pointer; }
  .link { stroke:#555; stroke-opacity:.5; }
  #tooltip { position:absolute; pointer-events:none; background:#222;
             padding:6px 10px; border-radius:4px; font-size:12px; display:none; }
</style>
</head>
<body>
<div id="tooltip"></div>
<div id="panel"><h3>Click a node to inspect</h3><div id="info">Nothing selected.</div></div>
<svg id="net"></svg>
<script>
const DATA = __GRAPH_DATA__;

const width = window.innerWidth - 320, height = window.innerHeight;
const svg = d3.select("#net").attr("width", width).attr("height", height);

// zoom + pan
const g = svg.append("g");
svg.call(d3.zoom().scaleExtent([0.3, 8]).on("zoom", e => g.attr("transform", e.transform)));

const color = d3.scaleOrdinal(d3.schemeCategory10);

const sim = d3.forceSimulation(DATA.nodes)
    .force("link", d3.forceLink(DATA.links).id(d => d.id).distance(60))
    .force("charge", d3.forceManyBody().strength(-120))
    .force("center", d3.forceCenter(width/2, height/2));

const link = g.append("g").selectAll("line").data(DATA.links).join("line")
    .attr("class", "link")
    .attr("stroke-width", d => Math.min(4, 0.5 + d.weight * 0.3));

const node = g.append("g").selectAll("circle").data(DATA.nodes).join("circle")
    .attr("class", "node")
    .attr("r", d => 4 + d.degree * 22)
    .attr("fill", d => color(d.community));

node.append("title").text(d => "Person " + d.id);

const tooltip = d3.select("#tooltip");
node.on("mouseover", (e, d) => {
      tooltip.style("display", "block")
             .style("left", (e.pageX+12)+"px").style("top", (e.pageY+12)+"px")
             .html(`<b>Person d.id</b><br>Degree:{d.id}</b><br>Degree:d.id</b><br>Degree:{d.degree}<br>Betweenness: d.betweenness<br>Community:{d.betweenness}<br>Community:d.betweenness<br>Community:{d.community}`);
    })
    .on("mouseout", () => tooltip.style("display","none"))
    .call(d3.drag()
        .on("start", (e,d)=>{ if(!e.active) sim.alphaTarget(.3).restart(); d.fx=d.x; d.fy=d.y; })
        .on("drag",   (e,d)=>{ d.fx=e.x; d.fy=e.y; })
        .on("end",    (e,d)=>{ if(!e.active) sim.alphaTarget(0); d.fx=null; d.fy=null; }));

node.on("click", (e, d) => {
    document.getElementById("info").innerHTML =
      `<h2 style='color:color(d.community)′>Person{color(d.community)}'>Personcolor(d.community)′>Person{d.id}</h2>` +
      `<p><b>Degree centrality:</b> ${d.degree}</p>` +
      `<p><b>Betweenness:</b> ${d.betweenness}</p>` +
      `<p><b>Community:</b> d.community({d.community} (d.community({color(d.community)})</p>` +
      `<p><i>High betweenness + moderate degree = potential intermediary profile.</i></p>`;
});

sim.on("tick", () => {
    link.attr("x1", d=>d.source.x).attr("y1", d=>d.source.y)
        .attr("x2", d=>d.target.x).attr("y2", d=>d.target.y);
    node.attr("cx", d=>d.x).attr("cy", d=>d.y);
});
</script>
</body>
</html>"""

with open("caviar_viz.html", "w") as f:
    f.write(html.replace("__GRAPH_DATA__", json.dumps(graph_data)))

print("Wrote caviar_viz.html")


Exported 107 nodes, 214 links
Wrote caviar_viz.html


In [ ]:
# Cell 12: Serve the HTML so you can view it
from google.colab import output
output.serve_kernel_port_as_window(8000) if False else None
import http.server, threading, functools

handler = functools.partial(http.server.SimpleHTTPRequestHandler,
                            directory="/content")
threading.Thread(
    target=http.server.HTTPServer(("127.0.0.1", 8000), handler).serve_forever,
    daemon=True).start()

from IPython.display import IFrame
IFrame(src="/tmp/placeholder.html", width=10, height=10)  # noop
print("""
✅ Server started on port 8000.

To view fullscreen:
  File menu → 'Download' → 'Download caviar_viz.html'
  Then double-click the downloaded file — it opens in your browser
  with full interactivity (works offline too!).
""")



✅ Server started on port 8000.

To view fullscreen:
  File menu → 'Download' → 'Download caviar_viz.html'
  Then double-click the downloaded file — it opens in your browser
  with full interactivity (works offline too!).



In [ ]:
# Cell 14: Generate synthetic anonymized demonstration network
import random, csv, os
from datetime import datetime, timedelta

random.seed(42)
os.makedirs("data", exist_ok=True)

N_NODES   = 220
N_EVENTS  = 2500
COMMUNITY_SIZES = [80, 55, 45, 40]          # 4 normal communities
ANOMALOUS_NODES = list(range(201, 211))     # P201–P210: injected anomalies

REL_TYPES = ["call", "message", "meeting", "transaction", "shared_location"]
start_date = datetime(2026, 8, 1)

def pid(i): return f"P{i:03d}"

nodes = [pid(i) for i in range(1, N_NODES + 1)]
community_of = {}
ci = 0
for size in COMMUNITY_SIZES:
    for n in nodes[ci:ci+size]:
        community_of[n] = ci
    ci += size
for n in ANOMALOUS_NODES:
    community_of[n] = len(COMMUNITY_SIZES)   # community 4 = anomaly group

events = []
t = 0
while len(events) < N_EVENTS - 120:
    # mostly intra-community edges (normal pattern)
    c = random.randrange(len(COMMUNITY_SIZES))
    members = [n for n, cc in community_of.items() if cc == c]
    a, b = random.sample(members, 2)
    ts = start_date + timedelta(hours=random.randrange(0, 60*24))
    rel = random.choices(REL_TYPES, weights=[35,30,15,12,8])[0]
    events.append((a, b, rel, ts.strftime("%Y-%m-%d %H:%M"),
                   random.randint(1, 9)))
    t += 1

# --- anomaly A: bridge brokers connect many communities ---
for i, broker in enumerate(["P201","P202","P203"]):
    for _ in range(25):
        other_c = random.randrange(len(COMMUNITY_SIZES))
        members = [n for n,cc in community_of.items() if cc==other_c and n not in ANOMALOUS_NODES]
        b = random.choice(members)
        ts = start_date + timedelta(hours=random.randrange(0, 60*24))
        events.append((broker, b, "transaction" if i==0 else "call",
                       ts.strftime("%Y-%m-%d %H:%M"), random.randint(5,9)))

# --- anomaly B: unusually dense hidden sub-cluster with burst timing ---
burst_day = start_date + timedelta(days=37)
for _ in range(45):
    a, b = random.sample(ANOMALOUS_NODES, 2)
    ts = burst_day + timedelta(minutes=random.randrange(0, 600))  # tight 10h window
    events.append((a, b, "meeting", ts.strftime("%Y-%m-%d %H:%M"), random.randint(7,10)))

with open("data/network_events.csv","w",newline="") as f:
    w = csv.writer(f); w.writerow(["source","target","relationship","timestamp","weight"])
    w.writerows(events)

with open("data/network_nodes.csv","w",newline="") as f:
    w = csv.writer(f); w.writerow(["entity_id","community_id","label"])
    for n in nodes:
        w.writerow([n, community_of[n], "SYNTHETIC/ANONYMIZED"])

print(f"Wrote data/network_events.csv : {len(events)} events")
print(f"Wrote data/network_nodes.csv  : {len(nodes)} nodes")
print("LABEL: Synthetic/Anonymized Demonstration Dataset — not real individuals.")


ValueError: Sample larger than population or is negative

In [ ]:
# Cell 14 (fixed): Generate synthetic anonymized demonstration network
import random, csv, os
from datetime import datetime, timedelta

random.seed(42)
os.makedirs("data", exist_ok=True)

N_NODES  = 220
N_EVENTS = 2500
COMMUNITY_SIZES = [80, 55, 45, 40]
ANOMALOUS_NODES = [f"P{i:03d}" for i in range(201, 211)]   # P201-P210

REL_TYPES = ["call", "message", "meeting", "transaction", "shared_location"]
start_date = datetime(2026, 8, 1)
def pid(i): return f"P{i:03d}"

nodes = [pid(i) for i in range(1, N_NODES + 1)]

# assign communities, but keep anomalous nodes OUT of normal communities
community_of, members_by_comm = {}, {}
idx = 0
for ci, size in enumerate(COMMUNITY_SIZES):
    members_by_comm[ci] = []
    for n in nodes[idx:idx+size]:
        if n not in ANOMALOUS_NODES:
            community_of[n] = ci
            members_by_comm[ci].append(n)
    idx += size
community_of_len = len(COMMUNITY_SIZES)
for n in ANOMALOUS_NODES:
    community_of[n] = community_of_len          # community 4 = anomaly group
members_by_comm[community_of_len] = ANOMALOUS_NODES

events = []
while len(events) < N_EVENTS - 120:
    c = random.randrange(community_of_len)
    members = members_by_comm[c]
    if len(members) < 2:                        # guard
        continue
    a, b = random.sample(members, 2)
    ts = start_date + timedelta(hours=random.randrange(0, 60*24))
    rel = random.choices(REL_TYPES, weights=[35,30,15,12,8])[0]
    events.append((a, b, rel, ts.strftime("%Y-%m-%d %H:%M"), random.randint(1, 9)))

# anomaly A: bridge brokers reaching into every community
for i, broker in enumerate(["P201","P202","P203"]):
    for _ in range(25):
        c = random.randrange(community_of_len)
        members = [m for m in members_by_comm[c] if m not in ANOMALOUS_NODES]
        if not members:
            continue
        b = random.choice(members)
        ts = start_date + timedelta(hours=random.randrange(0, 60*24))
        events.append((broker, b, "transaction" if i == 0 else "call",
                       ts.strftime("%Y-%m-%d %H:%M"), random.randint(5, 9)))

# anomaly B: dense hidden sub-cluster bursting in a tight 10-hour window
burst_day = start_date + timedelta(days=37)
for _ in range(45):
    a, b = random.sample(ANOMALOUS_NODES, 2)
    ts = burst_day + timedelta(minutes=random.randrange(0, 600))
    events.append((a, b, "meeting", ts.strftime("%Y-%m-%d %H:%M"), random.randint(7, 10)))

with open("data/network_events.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["source","target","relationship","timestamp","weight"])
    w.writerows(events)

with open("data/network_nodes.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["entity_id","community_id","label"])
    for n in nodes:
        w.writerow([n, community_of[n], "SYNTHETIC/ANONYMIZED"])

print(f"Wrote data/network_events.csv : {len(events)} events")
print(f"Wrote data/network_nodes.csv  : {len(nodes)} nodes")
print("LABEL: Synthetic/Anonymized Demonstration Dataset — not real individuals.")


Wrote data/network_events.csv : 2500 events
Wrote data/network_nodes.csv  : 220 nodes
LABEL: Synthetic/Anonymized Demonstration Dataset — not real individuals.


In [ ]:
# Cell 15: Build graph and compute full feature table for all 220 entities
import pandas as pd, networkx as nx
from networkx.algorithms.community import louvain_communities
from collections import defaultdict

events = pd.read_csv("data/network_events.csv")
nodes  = pd.read_csv("data/network_nodes.csv")

G = nx.Graph()
for _, row in nodes.iterrows():
    G.add_node(row["entity_id"], true_community=row["community_id"])

rel_counts = defaultdict(lambda: defaultdict(int))
for _, e in events.iterrows():
    a, b = e["source"], e["target"]
    rel_counts[a][e["relationship"]] += 1
    rel_counts[b][e["relationship"]] += 1
    if G.has_edge(a, b):
        G[a][b]["weight"] += e["weight"]
    else:
        G.add_edge(a, b, weight=e["weight"])

# detected communities (from structure, NOT the planted labels — important honesty point)
detected = louvain_communities(G, weight="weight", seed=42)
node_comm = {}
for ci, comm in enumerate(detected):
    for n in comm: node_comm[n] = ci

deg  = nx.degree_centrality(G)
bet  = nx.betweenness_centrality(G, weight="weight")
clo  = nx.closeness_centrality(G)
eig  = nx.eigenvector_centrality(G, weight="weight", max_iter=1000)
wdeg = dict(G.degree(weight="weight"))

# temporal activity: events per active day, max burst in one day
events["date"] = events["timestamp"].str[:10]
daily = defaultdict(lambda: defaultdict(int))
for _, e in events.iterrows():
    daily[e["source"]][e["date"]] += 1
    daily[e["target"]][e["date"]] += 1

rows = []
for n in G.nodes():
    rc = rel_counts[n]
    days = daily[n]
    rows.append({
        "entity_id": n,
        "degree": G.degree(n),
        "degree_centrality": round(deg[n], 4),
        "betweenness": round(bet[n], 5),
        "closeness": round(clo[n], 4),
        "eigenvector": round(eig[n], 6),
        "weighted_degree": wdeg[n],
        "unique_contacts": G.degree(n),
        "n_calls": rc.get("call", 0),
        "n_messages": rc.get("message", 0),
        "n_meetings": rc.get("meeting", 0),
        "n_transactions": rc.get("transaction", 0),
        "n_locations": rc.get("shared_location", 0),
        "community": node_comm[n],
        "active_days": len(days),
        "max_daily_burst": max(days.values()),
        "neighbour_avg_degree": round(sum(G.degree(nb) for nb in G.neighbors(n)) / max(1, G.degree(n)), 2),
    })

feat = pd.DataFrame(rows)
feat.to_csv("data/node_features.csv", index=False)

print(f"Feature table: {feat.shape[0]} entities x {feat.shape[1]-1} features")
print()
print("Detected communities:", [len(c) for c in detected])
print()
print("=== TOP 10 BY BETWEENNESS (bridge candidates) ===")
print(feat.nlargest(10, "betweenness")[["entity_id","degree","betweenness","community","n_transactions"]].to_string(index=False))
print()
print("=== TOP 10 BY DAILY BURST (temporal-anomaly candidates) ===")
print(feat.nlargest(10, "max_daily_burst")[["entity_id","max_daily_burst","active_days","community"]].to_string(index=False))


Feature table: 220 entities x 16 features

Detected communities: [80, 55, 45, 30, 10]

=== TOP 10 BY BETWEENNESS (bridge candidates) ===
entity_id  degree  betweenness  community  n_transactions
     P201      29      0.39091          4              25
     P203      28      0.18349          4               0
     P202      30      0.15247          4               0
     P004      17      0.12304          0               2
     P219      25      0.10890          3               4
     P050      11      0.10231          0               0
     P047      17      0.09425          0               2
     P091      19      0.09036          1               4
     P181      21      0.08884          3               3
     P125      19      0.08351          1               5

=== TOP 10 BY DAILY BURST (temporal-anomaly candidates) ===
entity_id  max_daily_burst  active_days  community
     P210               14            1          4
     P202               12           20          4
     P207  

In [ ]:
# Cell 16a: Install PyTorch Geometric
!pip install -q torch_geometric


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.8 MB/s eta 0:00:00


In [ ]:
# Cell 16b: Train GraphSAGE, score anomalies, export results
import pandas as pd, numpy as np, json, torch, torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from sklearn.preprocessing import StandardScaler

feat = pd.read_csv("data/node_features.csv")
events = pd.read_csv("data/network_events.csv")

# --- feature matrix ---
FEATURE_COLS = ["degree_centrality","betweenness","closeness","eigenvector",
                "weighted_degree","active_days","max_daily_burst",
                "neighbour_avg_degree","n_calls","n_messages","n_meetings",
                "n_transactions","n_locations"]
X = StandardScaler().fit_transform(feat[FEATURE_COLS].values)
x = torch.tensor(X, dtype=torch.float)

# --- edge index from events ---
ids = feat["entity_id"].tolist()
idx = {e: i for i, e in enumerate(ids)}
edge_list = [(idx[a], idx[b]) for a, b in zip(events["source"], events["target"])]
edge_index = torch.tensor(list(zip(*edge_list)), dtype=torch.long)
data = Data(x=x, edge_index=edge_index)

# --- GraphSAGE autoencoder-style model ---
class GraphSAGEEncoder(torch.nn.Module):
    def __init__(self, dim_in, hid=32, out=8):
        super().__init__()
        self.s1 = SAGEConv(dim_in, hid)
        self.s2 = SAGEConv(hid, out)
    def forward(self, x, ei):
        return F.relu(self.s2(F.relu(self.s1(x, ei)), ei))

class Decoder(torch.nn.Module):
    def __init__(self, emb=8, dim_out=len(FEATURE_COLS)):
        super().__init__()
        self.lin1 = torch.nn.Linear(emb, 16)
        self.lin2 = torch.nn.Linear(16, dim_out)
    def forward(self, z):
        return self.lin2(F.relu(self.lin1(z)))

torch.manual_seed(42)
enc, dec = GraphSAGEEncoder(len(FEATURE_COLS)), Decoder()
opt = torch.optim.Adam(list(enc.parameters()) + list(dec.parameters()), lr=0.01)

for epoch in range(200):
    opt.zero_grad()
    z = enc(data.x, data.edge_index)
    recon = dec(z)
    loss = F.mse_loss(recon, data.x)
    loss.backward(); opt.step()
print(f"Trained GraphSAGE: final reconstruction MSE = {loss.item():.4f}")

with torch.no_grad():
    z = enc(data.x, data.edge_index)
    err = ((dec(z) - data.x) ** 2).mean(dim=1).numpy()

# --- scale errors to 0-100 anomaly score ---
smin, smax = err.min(), err.max()
score = 100 * (err - smin) / max(smax - smin, 1e-9)

feat["anomaly_score"] = np.round(score, 1).astype(int)
def band(s): return ("LOW" if s <= 30 else "MODERATE" if s <= 60
                     else "HIGH" if s <= 80 else "VERY HIGH")
feat["band"] = feat["anomaly_score"].apply(band)

print("\n=== TOP 15 ANOMALOUS ENTITIES (GraphSAGE) ===")
top = feat.nlargest(15, "anomaly_score")
print(top[["entity_id","anomaly_score","band","degree","betweenness",
           "community","max_daily_burst"]].to_string(index=False))

planted = {f"P{i:03d}" for i in range(201, 211)}
in_top = sum(e.startswith(("P20","P21")) and e in planted for e in top["entity_id"])
print(f"\nPlanted anomalies found in top 15: {in_top}/10")

# --- export for frontend ---
results = {
    "_label": "Synthetic/Anonymized Demonstration Dataset — scores are network "
              "anomaly / investigation-priority indicators, NOT guilt predictions.",
    "nodes": [
        {"id": r.entity_id, "anomaly": int(r.anomaly_score), "band": r.band,
         "degree": int(r.degree), "betweenness": float(r.betweenness),
         "community": int(r.community)}
        for r in feat.itertuples()
    ],
    "links": [
        {"source": a, "target": b}
        for a, b in zip(events["source"], events["target"])
    ],
}
json.dump(results, open("gnn_results.json", "w"))
print("\nWrote gnn_results.json")


Trained GraphSAGE: final reconstruction MSE = 0.0769

=== TOP 15 ANOMALOUS ENTITIES (GraphSAGE) ===
entity_id  anomaly_score      band  degree  betweenness  community  max_daily_burst
     P181            100 VERY HIGH      21      0.08884          3                2
     P015             51  MODERATE      21      0.02246          0                2
     P109             48  MODERATE      21      0.00025          1                3
     P050             44  MODERATE      11      0.10231          0                1
     P179             38  MODERATE      26      0.00312          2                2
     P183             38  MODERATE      20      0.01182          3                3
     P101             37  MODERATE      15      0.00048          1                2
     P215             37  MODERATE      25      0.00263          3                4
     P185             36  MODERATE      22      0.00043          3                5
     P167             34  MODERATE      22      0.00286     

In [ ]:
# Cell 17: Hybrid anomaly score = embedding-space outlier + structural indicators
import numpy as np, pandas as pd, json
import torch
from sklearn.preprocessing import StandardScaler, MinMaxScaler

feat = pd.read_csv("data/node_features.csv")
FEATURE_COLS = ["degree_centrality","betweenness","closeness","eigenvector",
                "weighted_degree","active_days","max_daily_burst",
                "neighbour_avg_degree","n_calls","n_messages","n_meetings",
                "n_transactions","n_locations"]

# --- retrain encoder quickly and keep embeddings ---
X = StandardScaler().fit_transform(feat[FEATURE_COLS].values)
x = torch.tensor(X, dtype=torch.float)
events = pd.read_csv("data/network_events.csv")
ids = feat["entity_id"].tolist(); idx = {e:i for i,e in enumerate(ids)}
ei = torch.tensor(list(zip(*[(idx[a],idx[b]) for a,b in zip(events["source"],events["target"])])), dtype=torch.long)

from torch_geometric.nn import SAGEConv
import torch.nn.functional as F
torch.manual_seed(42)

class Enc(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.s1, self.s2 = SAGEConv(len(FEATURE_COLS),32), SAGEConv(32,8)
    def forward(self,x,ei): return F.relu(self.s2(F.relu(self.s1(x,ei)),ei))

enc = Enc(); opt = torch.optim.Adam(enc.parameters(), lr=0.01)
for ep in range(200):
    opt.zero_grad()
    # unsupervised objective: embeddings should predict each node's own scaled features
    loss = F.mse_loss(enc(x, ei), x[:, :8])
    loss.backward(); opt.step()

with torch.no_grad():
    Z = enc(x, ei).numpy()          # 220 x 8 embeddings

# --- component 1: embedding-space outlier score ---
# distance from each node's centroid within its detected community
comm_centroids = {}
for c in feat["community"].unique():
    m = feat["community"] == c
    comm_centroids[c] = Z[m.values].mean(axis=0)
emb_outlier = np.array([
    np.linalg.norm(Z[i] - comm_centroids[c])
    for i, c in enumerate(feat["community"])
])

# --- components 2-4: structural signals (already computed) ---
mm = MinMaxScaler()
s_between = mm.fit_transform(feat[["betweenness"]]).ravel()
s_burst   = mm.fit_transform(feat[["max_daily_burst"]]).ravel()
s_emb     = MinMaxScaler().fit_transform(emb_outlier.reshape(-1,1)).ravel()

# --- weighted blend (weights are design choices; state them openly) ---
W_EMB, W_BET, W_BURST = 0.5, 0.3, 0.2
score = 100*(W_EMB*s_emb + W_BET*s_between + W_BURST*s_burst)
feat["anomaly_score"] = np.round(score).astype(int)
feat["band"] = feat["anomaly_score"].apply(
    lambda s: "LOW" if s<=30 else "MODERATE" if s<=60 else "HIGH" if s<=80 else "VERY HIGH")

top = feat.nlargest(15, "anomaly_score")
print(top[["entity_id","anomaly_score","band","degree","betweenness",
           "community","max_daily_burst"]].to_string(index=False))

planted = {f"P{i:03d}" for i in range(201,211)}
hits = sum(e in planted for e in top["entity_id"])
print(f"\nPlanted anomalies in top 15: {hits}/10")

# per-component breakdown for explainability panel
for _, r in top.head(5).iterrows():
    i = feat.index[feat.entity_id==r.entity_id][0]
    print(f"{r.entity_id}: emb-outlier={s_emb[i]:.2f} betweenness={s_between[i]:.2f} burst={s_burst[i]:.2f}")

results = {
 "_label":"Synthetic/Anonymized Demonstration Dataset — network anomaly / investigation-priority scores, NOT guilt predictions.",
 "_method":"Hybrid: GraphSAGE embedding-community-outlier (w=0.5) + betweenness (w=0.3) + temporal burst (w=0.2)",
 "nodes":[{"id":r.entity_id,"anomaly":int(r.anomaly_score),"band":r.band,
           "degree":int(r.degree),"betweenness":float(r.betweenness),
           "community":int(r.community),
           "emb":round(float(s_emb[i]),3),"bet_c":round(float(s_between[i]),3),
           "burst_c":round(float(s_burst[i]),3)}
          for i,r in feat.iterrows()],
 "links":[{"source":a,"target":b} for a,b in zip(events["source"],events["target"])],
}
json.dump(results, open("gnn_results.json","w"))
print("\nWrote gnn_results.json")


entity_id  anomaly_score      band  degree  betweenness  community  max_daily_burst
     P201             83 VERY HIGH      29      0.39091          4                8
     P202             75      HIGH      30      0.15247          4               12
     P203             70      HIGH      28      0.18349          4                5
     P210             62      HIGH       8      0.00002          4               14
     P204             40  MODERATE       4      0.00000          4                5
     P205             40  MODERATE       7      0.00836          4               11
     P207             39  MODERATE       8      0.00031          4               12
     P209             38  MODERATE       4      0.00000          4                6
     P208             32  MODERATE       5      0.00000          4                8
     P142             31  MODERATE      28      0.01142          2                4
     P065             29       LOW      24      0.01817          0          

In [ ]:
# Cell 18: Build final investigator frontend
import json, pandas as pd

results = json.load(open("gnn_results.json"))
events = pd.read_csv("data/network_events.csv")

# aggregate links by pair+type for filtering
links = {}
for _, e in events.iterrows():
    key = tuple(sorted([e["source"], e["target"]]))
    links.setdefault(key, {"source": key[0], "target": key[1],
                           "weight": 0, "types": set()})
    links[key]["weight"] += e["weight"]
    links[key]["types"].add(e["relationship"])
for v in links.values(): v["types"] = sorted(v["types"])
link_list = [{"source":v["source"],"target":v["target"],
              "weight":v["weight"],"types":v["types"]} for v in links.values()]

payload = {"nodes": results["nodes"], "links": link_list,
           "label": results["_label"]}
html = """<!DOCTYPE html>
<html><head><meta charset="utf-8">
<title>SIH26189 — Criminal Network Analysis</title>
<script src="https://d3js.org/d3.v7.min.js"></script>
<style>
body{margin:0;font-family:sans-serif;background:#0d1117;color:#e6edf3;display:flex;height:100vh}
#left{flex:1;position:relative}
#panel{width:330px;background:#161b22;border-left:2px solid #30363d;padding:14px;overflow-y:auto;box-sizing:border-box}
.controls{position:absolute;top:10px;left:10px;z-index:5;display:flex;gap:8px;align-items:center}
select,input{background:#21262d;color:#e6edf3;border:1px solid #30363d;padding:4px 8px;border-radius:4px}
.node{stroke:#000;cursor:pointer}.link{stroke-opacity:.35}
#toplist div{padding:2px 6px;cursor:pointer;border-radius:3px}#toplist div:hover{background:#21262d}
.tick{color:#3fb950}h3,h4{margin:.4em 0}.badge{font-size:11px;padding:2px 8px;border-radius:10px;color:#fff}
.LOW{background:#238636}.MODERATE{background:#9e6a03}.HIGH{background:#da3633}.VERYHIGH{background:#f85149}
#legend{position:absolute;bottom:10px;left:10px;font-size:12px;background:#161b22cc;padding:8px;border-radius:6px}
.dot{display:inline-block;width:10px;height:10px;border-radius:50%;margin-right:4px}
#disclaimer{font-size:10px;color:#8b949e;margin-top:12px;border-top:1px solid #30363d;padding-top:8px}
</style></head><body>
<div id="left">
 <div class="controls">
   <input id="search" placeholder="Search e.g. P201" size="10">
   <select id="relFilter"><option value="">All relationships</option>
     <option>call</option><option>message</option><option>meeting</option>
     <option>transaction</option><option>shared_location</option></select>
   <select id="commFilter"><option value="">All communities</option></select>
   <button id="resetBtn">Reset</button>
 </div>
 <div id="toplist" style="position:absolute;top:44px;left:10px;z-index:5;background:#161b22cc;padding:6px;border-radius:6px;font-size:12px"></div>
 <svg id="net" width="100%" height="100%"></svg>
 <div id="tooltip" style="position:absolute;display:none;background:#222;padding:6px 10px;border-radius:4px;font-size:12px;pointer-events:none"></div>
 <div id="legend">
   <span class="dot" style="background:#58a6ff"></span>Low anomaly
   <span class="dot" style="background:#d29922"></span>Moderate
   <span class="dot" style="background:#db6d28"></span>High
   <span class="dot" style="background:#f85149"></span>Very high<br>
   Node size = anomaly score &nbsp;|&nbsp; click node → highlight connections
 </div>
</div>
<div id="panel">
 <h3>INVESTIGATOR PANEL</h3><div id="info">Select an entity.</div>
 <div style="margin-top:10px"><b>Top anomalous entities:</b><div id="top10"></div></div>
 <div id="disclaimer">Synthetic/Anonymized Demonstration Dataset.<br>Scores indicate NETWORK ANOMALY / INVESTIGATION PRIORITY only. This system does not determine guilt. Human investigators make all determinations.</div>
</div>
<script>
const DATA=__DATA__;
const W=document.getElementById("left").clientWidth,H=window.innerHeight;
const svg=d3.select("#net"),g=svg.append("g");
svg.call(d3.zoom().scaleExtent([0.3,8]).on("zoom",e=>g.attr("transform",e.transform)));
const col=s=>s==="LOW"?"#58a6ff":s==="MODERATE"?"#d29922":s==="HIGH"?"#db6d28":"#f85149";
const sim=d3.forceSimulation(DATA.nodes)
 .force("link",d3.forceLink(DATA.links).id(d=>d.id).distance(50))
 .force("charge",d3.forceManyBody().strength(-90))
 .force("center",d3.forceCenter(W/2,H/2));
const link=g.append("g").selectAll("line").data(DATA.links).join("line")
 .attr("class","link").attr("stroke","#555")
 .attr("stroke-width",d=>Math.min(4,.4+d.weight*.15));
const node=g.append("g").selectAll("circle").data(DATA.nodes).join("circle")
 .attr("class","node").attr("r",d=>3+d.anomaly*.06)
 .attr("fill",d=>col(d.band));
node.append("title").text(d=>d.id+" — "+d.band);
const tip=d3.select("#tooltip");
node.on("mouseover",(e,d)=>tip.style("display","block").style("left",(e.offsetX+12)+"px").style("top",(e.offsetY+12)+"px")
  .html(`<b>d.id</b>⋅score{d.id}</b> · scored.id</b>⋅score{d.anomaly} · ${d.band}`)).on("mouseout",()=>tip.style("display","none"))
 .call(d3.drag().on("start",(e,d)=>{if(!e.active)sim.alphaTarget(.3).restart();d.fx=d.x;d.fy=d.y})
  .on("drag",(e,d)=>{d.fx=e.x;d.fy=e.y}).on("end",(e,d)=>{if(!e.active)sim.alphaTarget(0);d.fx=d.fy=null}));

const nbr={}; DATA.links.forEach(l=>{
 (nbr[l.source.id||l.source]=nbr[l.source.id||l.source]||[]).push(l.target.id||l.target);
 (nbr[l.target.id||l.target]=nbr[l.target.id||l.target]||[]).push(l.source.id||l.source);});
function neighbors(id,n){let cur=new Set([id]);for(let k=0;k<n;k++){const nx=new Set(cur);
 cur.forEach(c=>(nbr[c]||[]).forEach(x=>nx.add(x)));cur=nx;}return cur;}

function show(id){
 const d=DATA.nodes.find(n=>n.id===id); if(!d)return;
 const ns=nbr[id]||[];
 let ind=[];
 if(d.degree>=25)ind.push("✓ High degree centrality ("+d.degree+")");
 if(d.betweenness>=.08)ind.push("✓ High betweenness — bridges communities");
 if(d.bet_c>=.5)ind.push("✓ Unusual structural position (embedding outlier "+d.emb.toFixed(2)+")");
 if(d.burst_c>=.5)ind.push("✓ Temporal interaction anomaly (burst component "+d.burst_c.toFixed(2)+")");
 if(!ind.length)ind=["No strong individual indicators — routine pattern"];
 document.getElementById("info").innerHTML=
  `<h2>${d.id}</h2><p>ANONYMIZED ENTITY</p>`+
  `<span class="badge {d.band.replace(' ','')}">{d.band}</span>`+
  `<h3>Network Anomaly Score: ${d.anomaly}/100</h3>`+
  `<table>`+
  `<tr><td>Degree:</td><td>${d.degree}</td></tr>`+
  `<tr><td>Betweenness:</td><td>${d.betweenness}</td></tr>`+
  `<tr><td>Community:</td><td>${d.community}</td></tr>`+
  `<tr><td>Unique contacts:</td><td>${ns.length}</td></tr>`+
  `<tr><td>Embedding outlier:</td><td>${d.emb.toFixed(2)}</td></tr>`+
  `<tr><td>Burst component:</td><td>${d.burst_c.toFixed(2)}</td></tr></table>`+
  `<h4>AI indicators:</h4><ul>{ind.map(i=>`<li class="tick">{i}</li>`).join("")}</ul>`+
  `<p><b>Recommendation:</b> Prioritize for human investigation review.</p>`;
 // highlight neighborhood
 node.attr("opacity",n=>neighbors(id,1).has(n.id)?1:.12)
     .attr("stroke",n=>n.id===id?"#fff":null).attr("stroke-width",n=>n.id===id?2:0);
 link.attr("opacity",l=>l.source.id===id||l.target.id===id?1:.05);
}
function clearSel(){node.attr("opacity",1).attr("stroke",null).attr("stroke-width",0);link.attr("opacity",null);}
node.on("click",(e,d)=>show(d.id));
document.getElementById("resetBtn").onclick=()=>{clearSel();
 document.getElementById("relFilter").value="";document.getElementById("commFilter").value="";
 document.getElementById("search").value="";applyFilters();};

function applyFilters(){
 const rel=document.getElementById("relFilter").value;
 const com=document.getElementById("commFilter").value;
 const q=document.getElementById


_IncompleteInputError: incomplete input (289754475.py, line 21)

In [ ]:
# Cell 18a: investigator.html — PART 1
part1 = """<!DOCTYPE html>
<html><head><meta charset="utf-8">
<title>SIH26189 - Criminal Network Analysis</title>
<script src="https://d3js.org/d3.v7.min.js"></script>
<style>
body{margin:0;font-family:sans-serif;background:#0d1117;color:#e6edf3;display:flex;height:100vh}
#left{flex:1;position:relative}
#panel{width:330px;background:#161b22;border-left:2px solid #30363d;padding:14px;overflow-y:auto;box-sizing:border-box}
.controls{position:absolute;top:10px;left:10px;z-index:5;display:flex;gap:8px}
select,input{background:#21262d;color:#e6edf3;border:1px solid #30363d;padding:4px 8px;border-radius:4px}
.node{stroke:#000;cursor:pointer}.link{stroke-opacity:.35}
.tick{color:#3fb950}h3,h4{margin:.4em 0}
.badge{font-size:11px;padding:2px 8px;border-radius:10px;color:#fff}
.LOW{background:#238636}.MODERATE{background:#9e6a03}.HIGH{background:#da3633}.VERYHIGH{background:#f85149}
#toplist div{padding:2px 6px;cursor:pointer}#toplist div:hover{background:#21262d}
.dot{display:inline-block;width:10px;height:10px;border-radius:50%;margin-right:4px}
#legend{position:absolute;bottom:10px;left:10px;font-size:12px;background:#161b22cc;padding:8px;border-radius:6px}
#disclaimer{font-size:10px;color:#8b949e;margin-top:12px;border-top:1px solid #30363d;padding-top:8px}
</style></head><body>
<div id="left">
 <div class="controls">
   <input id="search" placeholder="Search e.g. P201" size="10">
   <select id="relFilter"><option value="">All relationships</option>
     <option>call</option><option>message</option><option>meeting</option>
     <option>transaction</option><option>shared_location</option></select>
   <select id="commFilter"><option value="">All communities</option></select>
   <button id="resetBtn">Reset</button>
 </div>
 <div id="toplist" style="position:absolute;top:44px;left:10px;z-index:5;background:#161b22cc;padding:6px;border-radius:6px;font-size:12px"></div>
 <svg id="net" width="100%" height="100%"></svg>
 <div id="tooltip" style="position:absolute;display:none;background:#222;padding:6px 10px;border-radius:4px;font-size:12px;pointer-events:none"></div>
 <div id="legend">
   <span class="dot" style="background:#58a6ff"></span>Low
   <span class="dot" style="background:#d29922"></span>Moderate
   <span class="dot" style="background:#db6d28"></span>High
   <span class="dot" style="background:#f85149"></span>Very high<br>
   Node size = anomaly score | click node = highlight connections
 </div>
</div>
<div id="panel">
 <h3>INVESTIGATOR PANEL</h3><div id="info">Select an entity.</div>
 <div style="margin-top:10px"><b>Top anomalous entities:</b><div id="top10"></div></div>
 <div id="disclaimer">Synthetic/Anonymized Demonstration Dataset.<br>Scores indicate NETWORK ANOMALY / INVESTIGATION PRIORITY only. This system does not determine guilt.</div>
</div>
"""
with open("investigator_part1.txt", "w") as f:
    f.write(part1)
print("Part 1 written:", len(part1), "chars")


Part 1 written: 2789 chars


In [ ]:
# Cell 18b: investigator.html — PART 2 (JS) + assemble final file
import json, pandas as pd

part2 = """<script>
const DATA=__DATA__;
const W=document.getElementById("left").clientWidth,H=window.innerHeight;
const svg=d3.select("#net"),g=svg.append("g");
svg.call(d3.zoom().scaleExtent([0.3,8]).on("zoom",e=>g.attr("transform",e.transform)));
const col=s=>s==="LOW"?"#58a6ff":s==="MODERATE"?"#d29922":s==="HIGH"?"#db6d28":"#f85149";
const sim=d3.forceSimulation(DATA.nodes)
 .force("link",d3.forceLink(DATA.links).id(d=>d.id).distance(50))
 .force("charge",d3.forceManyBody().strength(-90))
 .force("center",d3.forceCenter(W/2,H/2));
const link=g.append("g").selectAll("line").data(DATA.links).join("line")
 .attr("class","link").attr("stroke","#555")
 .attr("stroke-width",d=>Math.min(4,.4+d.weight*.15));
const node=g.append("g").selectAll("circle").data(DATA.nodes).join("circle")
 .attr("class","node").attr("r",d=>3+d.anomaly*.06)
 .attr("fill",d=>col(d.band));
node.append("title").text(d=>d.id+" - "+d.band);
const tip=d3.select("#tooltip");
node.on("mouseover",(e,d)=>tip.style("display","block")
  .style("left",(e.offsetX+12)+"px").style("top",(e.offsetY+12)+"px")
  .html("<b>"+d.id+"</b> score "+d.anomaly+" "+d.band))
 .on("mouseout",()=>tip.style("display","none"))
 .call(d3.drag()
  .on("start",(e,d)=>{if(!e.active)sim.alphaTarget(.3).restart();d.fx=d.x;d.fy=d.y})
  .on("drag",(e,d)=>{d.fx=e.x;d.fy=e.y})
  .on("end",(e,d)=>{if(!e.active)sim.alphaTarget(0);d.fx=d.fy=null}));

const nbr={};
DATA.links.forEach(l=>{
 const s=l.source.id||l.source,t=l.target.id||l.target;
 (nbr[s]=nbr[s]||[]).push(t);(nbr[t]=nbr[t]||[]).push(s);});

function show(id){
 const d=DATA.nodes.find(n=>n.id===id); if(!d)return;
 const ns=nbr[id]||[];
 let ind=[];
 if(d.degree>=25)ind.push("High degree centrality ("+d.degree+" contacts)");
 if(d.betweenness>=.08)ind.push("High betweenness - bridges communities ("+d.betweenness+")");
 if(d.emb>=.5)ind.push("Embedding outlier - unusual position for its community ("+d.emb.toFixed(2)+")");
 if(d.burst_c>=.5)ind.push("Temporal interaction anomaly ("+d.burst_c.toFixed(2)+")");
 if(!ind.length)ind=["No strong individual indicators - routine pattern"];
 document.getElementById("info").innerHTML=
  "<h2>"+d.id+"</h2><p>ANONYMIZED ENTITY</p>"+
  '<span class="badge '+d.band.replace(' ','')+'">'+d.band+"</span>"+
  "<h3>Network Anomaly Score: "+d.anomaly+"/100</h3>"+
  "<table>"+
  "<tr><td>Degree:</td><td>"+d.degree+"</td></tr>"+
  "<tr><td>Betweenness:</td><td>"+d.betweenness+"</td></tr>"+
  "<tr><td>Community:</td><td>"+d.community+"</td></tr>"+
  "<tr><td>Unique contacts:</td><td>"+ns.length+"</td></tr>"+
  "<tr><td>Embedding outlier:</td><td>"+d.emb.toFixed(2)+"</td></tr>"+
  "<tr><td>Burst component:</td><td>"+d.burst_c.toFixed(2)+"</td></tr></table>"+
  "<h4>AI indicators:</h4><ul>"+ind.map(i=>'<li class="tick">'+i+"</li>").join("")+"</ul>"+
  "<p><b>Recommendation:</b> Prioritize for human investigation review.</p>";
 node.attr("opacity",n=>ns.includes(n.id)||n.id===id?1:.12)
     .attr("stroke",n=>n.id===id?"#fff":null)
     .attr("stroke-width",n=>n.id===id?2:0);
 link.attr("opacity",l=>{
   const s=l.source.id||l.source,t=l.target.id||l.target;
   return s===id||t===id?1:.05;});
}
function clearSel(){node.attr("opacity",1).attr("stroke",null).attr("stroke-width",0);
 link.attr("opacity",null);}
node.on("click",(e,d)=>show(d.id));

function applyFilters(){
 const rel=document.getElementById("relFilter").value;
 const com=document.getElementById("commFilter").value;
 const q=document.getElementById("search").value.trim().toUpperCase();
 const qActive=q.length>0;
 node.attr("display",d=>
   ((com===""||String(d.community)===com)&&(!qActive||d.id.startsWith(q)))?null:"none");
 link.attr("display",l=>{
   const s=l.source.id||l.source,t=l.target.id||l.target;
   if(rel && !(l.types||[]).includes(rel)) return "none";
   if(com!==""&&(String(DATA.nodes.find(n=>n.id===s).community)!==com)) return "none";
   if(qActive&&!s.startsWith(q)&&!t.startsWith(q)) return "none";
   return null;});
}
[...new Set(DATA.nodes.map(n=>n.community))].sort((a,b)=>a-b)
 .forEach(c=>document.getElementById("commFilter")
 .insertAdjacentHTML("beforeend","<option>"+c+"</option>"));
document.getElementById("relFilter").onchange=()=>{clearSel();applyFilters()};
document.getElementByIdcommFilter").onchange=()=>{clearSel();applyFilters()};
document.getElementById("search").oninput=()=>{clearSel();applyFilters()};
document.getElementById("resetBtn").onclick=()=>{
 clearSel();
 document.getElementById("relFilter").value="";
 document.getElementById("commFilter").value="";
 document.getElementById("search").value="";
 applyFilters();};

const top=[...DATA.nodes].sort((a,b)=>b.anomaly-a.anomaly).slice(0,10);
const t10=top.map(n=>'<div onclick="show(\\''+n.id+'\\')">'+n.id+" - "+n.anomaly+
 ' <span class="badge '+n.band.replace(' ','')+'" style="float:right">'+n.band+"</span></div>").join("");
document.getElementById("top10").innerHTML=t10;
document.getElementById("toplist").innerHTML="<b>TOP RISK</b><br>"+
 top.slice(0,5).map(n=>"<div onclick='show(\""+n.id+"\")'>"+n.id+": "+n.anomaly+"</div>").join("");
window.show=show;

sim.on("tick",()=>{
 link.attr("x1",d=>d.source.x).attr("y1",d=>d.source.y)
     .attr("x2",d=>d.target.x).attr("y2",d=>d.target.y);
 node.attr("cx",d=>d.x).attr("cy",d=>d.y);});
</script></body></html>"""

results = json.load(open("gnn_results.json"))
events = pd.read_csv("data/network_events.csv")

links = {}
for _, e in events.iterrows():
    key = tuple(sorted([e["source"], e["target"]]))
    L = links.setdefault(key, {"source": key[0], "target": key[1],
                               "weight": 0, "types": set()})
    L["weight"] += e["weight"]; L["types"].add(e["relationship"])
for v in links.values(): v["types"] = sorted(v["types"])

payload = json.dumps({"nodes": results["nodes"],
                      "links": list(links.values()),
                      "label": results["_label"]})

html = open("investigator_part1.txt").read() + part2.replace("__DATA__", payload)
open("


SyntaxError: unterminated string literal (detected at line 126) (1789015616.py, line 126)

In [ ]:
# Cell 18b: investigator.html — JS PART A
jsA = """<script>
const DATA=__DATA__;
const W=document.getElementById("left").clientWidth,H=window.innerHeight;
const svg=d3.select("#net"),g=svg.append("g");
svg.call(d3.zoom().scaleExtent([0.3,8]).on("zoom",e=>g.attr("transform",e.transform)));
const col=s=>s==="LOW"?"#58a6ff":s==="MODERATE"?"#d29922":s==="HIGH"?"#db6d28":"#f85149";
const sim=d3.forceSimulation(DATA.nodes)
 .force("link",d3.forceLink(DATA.links).id(d=>d.id).distance(50))
 .force("charge",d3.forceManyBody().strength(-90))
 .force("center",d3.forceCenter(W/2,H/2));
const link=g.append("g").selectAll("line").data(DATA.links).join("line")
 .attr("class","link").attr("stroke","#555")
 .attr("stroke-width",d=>Math.min(4,.4+d.weight*.15));
const node=g.append("g").selectAll("circle").data(DATA.nodes).join("circle")
 .attr("class","node").attr("r",d=>3+d.anomaly*.06)
 .attr("fill",d=>col(d.band));
node.append("title").text(d=>d.id+" - "+d.band);
const tip=d3.select("#tooltip");
node.on("mouseover",(e,d)=>tip.style("display","block")
  .style("left",(e.offsetX+12)+"px").style("top",(e.offsetY+12)+"px")
  .html("<b>"+d.id+"</b> score "+d.anomaly+" "+d.band))
 .on("mouseout",()=>tip.style("display","none"))
 .call(d3.drag()
  .on("start",(e,d)=>{if(!e.active)sim.alphaTarget(.3).restart();d.fx=d.x;d.fy=d.y})
  .on("drag",(e,d)=>{d.fx=e.x;d.fy=e.y})
  .on("end",(e,d)=>{if(!e.active)sim.alphaTarget(0);d.fx=d.fy=null}));
const nbr={};
DATA.links.forEach(l=>{
 const s=l.source.id||l.source,t=l.target.id||l.target;
 (nbr[s]=nbr[s]||[]).push(t);(nbr[t]=nbr[t]||[]).push(s);});
"""
open("investigator_part1.txt","w").write(open("investigator_part1.txt").read())
open("investigator_js_a.txt","w").write(jsA)
print("JS Part A written:", len(jsA), "chars")


JS Part A written: 1545 chars


In [ ]:
# Cell 18c: investigator.html — JS PART B
jsB = """function show(id){
 const d=DATA.nodes.find(n=>n.id===id); if(!d)return;
 const ns=nbr[id]||[];
 let ind=[];
 if(d.degree>=25)ind.push("High degree centrality ("+d.degree+" contacts)");
 if(d.betweenness>=.08)ind.push("High betweenness - bridges communities ("+d.betweenness+")");
 if(d.emb>=.5)ind.push("Embedding outlier - unusual position in community ("+d.emb.toFixed(2)+")");
 if(d.burst_c>=.5)ind.push("Temporal interaction anomaly ("+d.burst_c.toFixed(2)+")");
 if(!ind.length)ind=["No strong individual indicators - routine pattern"];
 document.getElementById("info").innerHTML=
  "<h2>"+d.id+"</h2><p>ANONYMIZED ENTITY</p>"+
  '<span class="badge '+d.band.replace(' ','')+'">'+d.band+"</span>"+
  "<h3>Network Anomaly Score: "+d.anomaly+"/100</h3>"+
  "<table><tr><td>Degree:</td><td>"+d.degree+"</td></tr>"+
  "<tr><td>Betweenness:</td><td>"+d.betweenness+"</td></tr>"+
  "<tr><td>Community:</td><td>"+d.community+"</td></tr>"+
  "<tr><td>Unique contacts:</td><td>"+ns.length+"</td></tr>"+
  "<tr><td>Embedding outlier:</td><td>"+d.emb.toFixed(2)+"</td></tr>"+
  "<tr><td>Burst component:</td><td>"+d.burst_c.toFixed(2)+"</td></tr></table>"+
  "<h4>AI indicators:</h4><ul>"+ind.map(i=>'<li class="tick">'+i+"</li>").join("")+"</ul>"+
  "<p><b>Recommendation:</b> Prioritize for human investigation review.</p>";
 node.attr("opacity",n=>ns.includes(n.id)||n.id===id?1:.12)
     .attr("stroke",n=>n.id===id?"#fff":null)
     .attr("stroke-width",n=>n.id===id?2:0);
 link.attr("opacity",l=>{
   const s=l.source.id||l.source,t=l.target.id||l.target;
   return s===id||t===id?1:.05;});
}
function clearSel(){node.attr("opacity",1).attr("stroke",null).attr("stroke-width",0);
 link.attr("opacity",null);}
node.on("click",(e,d)=>show(d.id));
function applyFilters(){
 const rel=document.getElementById("relFilter").value;
 const com=document.getElementById("commFilter").value;
 const q=document.getElementById("search").value.trim().toUpperCase();
 const qA=q.length>0;
 node.attr("display",d=>((com===""||String(d.community)===com)&&(!qA||d.id.startsWith(q)))?null:"none");
 link.attr("display",l=>{
   const s=l.source.id||l.source,t=l.target.id||l.target;
   if(rel&&!(l.types||[]).includes(rel))return "none";
   if(com!==""&&String(DATA.nodes.find(n=>n.id===s).community)!==com)return "none";
   if(qA&&!s.startsWith(q)&&!t.startsWith(q))return "none";
   return null;});
}
[...new Set(DATA.nodes.map(n=>n.community))].sort((a,b)=>a-b)
 .forEach(c=>document.getElementById("commFilter")
 .insertAdjacentHTML("beforeend","<option>"+c+"</option>"));
document.getElementById("relFilter").onchange=()=>{clearSel();applyFilters()};
document.getElementById("commFilter").onchange=()=>{clearSel();applyFilters()};
document.getElementById("search").oninput=()=>{clearSel();applyFilters()};
document.getElementById("resetBtn").onclick=()=>{clearSel();
 document.getElementById("relFilter").value="";
 document.getElementById("commFilter").value="";
 document.getElementById("search").value="";
 applyFilters();};
const top=[...DATA.nodes].sort((a,b)=>b.anomaly-a.anomaly).slice(0,10);
document.getElementById("top10").innerHTML=top.map(n=>
 '<div onclick="show(\\''+n.id+'\\')">'+n.id+" - "+n.anomaly+
 ' <span class="badge '+n.band.replace(' ','')+'" style="float:right">'+n.band+"</span></div>").join("");
document.getElementById("toplist").innerHTML="<b>TOP RISK</b><br>"+
 top.slice(0,5).map(n=>'<div onclick="show(\\''+n


_IncompleteInputError: incomplete input (2769422534.py, line 2)

In [ ]:
# Cell 18c-i: JS Part B, first half
jsb1 = """function show(id){
 const d=DATA.nodes.find(n=>n.id===id); if(!d)return;
 const ns=nbr[id]||[];
 let ind=[];
 if(d.degree>=25)ind.push("High degree centrality ("+d.degree+" contacts)");
 if(d.betweenness>=.08)ind.push("High betweenness - bridges communities");
 if(d.emb>=.5)ind.push("Embedding outlier - unusual position ("+d.emb.toFixed(2)+")");
 if(d.burst_c>=.5)ind.push("Temporal interaction anomaly ("+d.burst_c.toFixed(2)+")");
 if(!ind.length)ind=["No strong individual indicators - routine pattern"];
 document.getElementById("info").innerHTML=
  "<h2>"+d.id+"</h2><p>ANONYMIZED ENTITY</p>"+
  '<span class="badge '+d.band.replace(' ','')+'">'+d.band+"</span>"+
  "<h3>Network Anomaly Score: "+d.anomaly+"/100</h3>"+
  "<table><tr><td>Degree:</td><td>"+d.degree+"</td></tr>"+
  "<tr><td>Betweenness:</td><td>"+d.betweenness+"</td></tr>"+
  "<tr><td>Community:</td><td>"+d.community+"</td></tr>"+
  "<tr><td>Unique contacts:</td><td>"+ns.length+"</td></tr>"+
  "<tr><td>Embedding outlier:</td><td>"+d.emb.toFixed(2)+"</td></tr>"+
  "<tr><td>Burst component:</td><td>"+d.burst_c.toFixed(2)+"</td></tr></table>"+
  "<h4>AI indicators:</h4><ul>"+ind.map(i=>'<li class="tick">'+i+"</li>").join("")+"</ul>"+
  "<p><b>Recommendation:</b> Prioritize for human investigation review.</p>";
 node.attr("opacity",n=>ns.includes(n.id)||n.id===id?1:.12)
     .attr("stroke",n=>n.id===id?"#fff":null)
     .attr("stroke-width",n=>n.id===id?2:0);
 link.attr("opacity",l=>{
   const s=l.source.id||l.source,t=l.target.id||l.target;
   return s===id||t===id?1:.05;});
}
function clearSel(){node.attr("opacity",1).attr("stroke",null).attr("stroke-width",0);
 link.attr("opacity",null);}
node.on("click",(e,d)=>show(d.id));
"""
open("investigator_js_b1.txt","w").write(jsb1)
print("JS B part 1 written:", len(jsb1), "chars")
# Cell 18c-ii: JS Part B, second half
jsb2 = """function applyFilters(){
 const rel=document.getElementById("relFilter").value;
 const com=document.getElementById("commFilter").value;
 const q=document.getElementById("search").value.trim().toUpperCase();
 const qA=q.length>0;
 node.attr("display",d=>((com===""||String(d.community)===com)&&(!qA||d.id.startsWith(q)))?null:"none");
 link.attr("display",l=>{
   const s=l.source.id||l.source,t=l.target.id||l.target;
   if(rel&&!(l.types||[]).includes(rel))return "none";
   if(com!==""&&String(DATA.nodes.find(n=>n.id===s).community)!==com)return "none";
   if(qA&&!s.startsWith(q)&&!t.startsWith(q))return "none";
   return null;});
}
[...new Set(DATA.nodes.map(n=>n.community))].sort((a,b)=>a-b)
 .forEach(c=>document.getElementById("commFilter")
 .insertAdjacentHTML("beforeend","<option>"+c+"</option>"));
document.getElementById("relFilter").onchange=()=>{clearSel();applyFilters()};
document.getElementById("commFilter").onchange=()=>{clearSel();applyFilters()};
document.getElementById("search").oninput=()=>{clearSel();applyFilters()};
document.getElementById("resetBtn").onclick=()=>{clearSel();
 document.getElementById("relFilter").value="";
 document.getElementById("commFilter").value="";
 document.getElementById("search").value="";
 applyFilters();};
const top=[...DATA.nodes].sort((a,b)=>b.anomaly-a.anomaly).slice(0,10);
const mk=n=>'<div onclick="show(\\''+n.id+'\\')">'+n.id+" - "+n.anomaly+
 ' <span class="badge '+n.band.replace(' ','')+'">'+n.band+"</span></div>";
document.getElementById("top10").innerHTML=top.map(mk).join("");
document.getElementById("toplist").innerHTML="<b>TOP RISK</b><br>"+
 top.slice(0,5).map(mk).join("");
window.show=show;
sim.on("tick",()=>{
 link.attr("x1",d=>d.source.x).attr("y1",d=>d.source.y)
     .attr("x2",d=>d.target.x).attr("y2",d=>d.target.y);
 node.attr("cx",d=>d.x).attr("cy",d=>d.y);});
</script></body></html>"""
open("investigator_js_b2.txt","w").write(jsb2)
print("JS B part 2 written:", len(jsb2), "chars")


JS B part 1 written: 1713 chars
JS B part 2 written: 1874 chars


In [ ]:
# Cell 18d: Assemble investigator.html
import json, pandas as pd

results = json.load(open("gnn_results.json"))
events = pd.read_csv("data/network_events.csv")

links = {}
for _, e in events.iterrows():
    key = tuple(sorted([e["source"], e["target"]]))
    L = links.setdefault(key, {"source": key[0], "target": key[1],
                               "weight": 0, "types": set()})
    L["weight"] += e["weight"]; L["types"].add(e["relationship"])
for v in links.values(): v["types"] = sorted(v["types"])

payload = json.dumps({"nodes": results["nodes"],
                      "links": list(links.values()),
                      "label": results["_label"]})

html = (open("investigator_part1.txt").read()
        + open("investigator_js_a.txt").read().replace("__DATA__", payload)
        + open("investigator_js_b1.txt").read()
        + open("investigator_js_b2.txt").read())

open("investigator.html","w").write(html)
print(f"Wrote investigator.html ({len(html)} chars)")


Wrote investigator.html (179517 chars)


In [34]:
# Cell 19: Sanity checks on investigator.html
html = open("investigator.html").read()

checks = {
    "<script": html.count("<script"),
    "</script>": html.count("</script>"),
    "d3.forceSimulation present": "forceSimulation" in html,
    "DATA embedded": '"nodes"' in html,
    "__DATA__ placeholder gone": "__DATA__" not in html,
    "ends with </html>": html.rstrip().endswith("</html>"),
    "onclick escaping sample": "show(\\'" in html or "show('" in html,
}
for k, v in checks.items():
    print(f"{k}: {v}")


<script: 1
</script>: 1
d3.forceSimulation present: True
DATA embedded: True
__DATA__ placeholder gone: True
ends with </html>: True
onclick escaping sample: True


In [35]:
# Cell 20: Serve and display in Colab
from google.colab import output
output.serve_kernel_port_as_window(8010)

import http.server, threading, functools
handler = functools.partial(http.server.SimpleHTTPRequestHandler, directory="/content")
threading.Thread(target=http.server.HTTPServer(("127.0.0.1", 8010), handler).serve_forever, daemon=True).start()
print("Server running on port 8010 — a window should pop up.")


Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

Server running on port 8010 — a window should pop up.


In [36]:
# Cell 21: Embed d3.js directly into the HTML (offline-proof)
import urllib.request

try:
    d3src = urllib.request.urlopen("https://d3js.org/d3.v7.min.js", timeout=15).read().decode()
    print(f"D3 downloaded: {len(d3src)} chars")
except Exception as ex:
    print("D3 download FAILED:", ex)
    raise SystemExit("Check Colab internet; re-run this cell.")

html = open("investigator.html").read()
old = '<script src="https://d3js.org/d3.v7.min.js"></script>'
assert old in html, "CDN script tag not found!"
html = html.replace(old, "<script>\n" + d3src + "\n</script>")
open("investigator_offline.html", "w").write(html)
print(f"Wrote investigator_offline.html ({len(html)} chars) — fully self-contained.")


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



D3 download FAILED: HTTP Error 403: Forbidden
Traceback (most recent call last):
  File "/tmp/ipykernel_1464/141219509.py", line 5, in <cell line: 0>
    d3src = urllib.request.urlopen("https://d3js.org/d3.v7.min.js", timeout=15).read().decode()
            ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/urllib/request.py", line 189, in urlopen
    return opener.open(url, data, timeout)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/urllib/request.py", line 495, in open
    response = meth(req, response)
  File "/usr/lib/python3.13/urllib/request.py", line 604, in http_response
    response = self.parent.error(
        'http', request, response, code, msg, hdrs)
  File "/usr/lib/python3.13/urllib/request.py", line 533, in error
    return self._call_chain(*args)
           ~~~~~~~~~~~~~~~~^^^^^^^
  File "/usr/lib/python3.13/urllib/request.py", line 466, in _call_chain
    result = func(*args)
  File "/usr/lib/pytho

TypeError: object of type 'NoneType' has no len()

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
